# Automated Chess Video to PGN Extraction Pipeline

This notebook provides an automated end-to-end machine learning workflow. Execute the cells sequentially to perform the following stages: **Data Acquisition -> Model Training -> Game State Extraction**.

> **OpenCV GUI Warning:** Executing cells containing `cv2.imshow` will initialize an external GUI window. Upon completion of the required interactions, strictly use the `Esc` key to terminate the window. **Do not force-close the window using the OS close button**, as this may result in a kernel deadlock.

## Execute Module: `01_auto_data_collector.py`

In [ ]:
import cv2
import numpy as np
import os

# ================= 1. Initialize output directories for 13 classes =================
save_dir = "chess_dataset"
classes = [
    "00_Empty", "01_P", "02_N", "03_B", "04_R", "05_Q", "06_K",
    "07_p_black", "08_n_black", "09_b_black", "10_r_black", "11_q_black", "12_k_black"
]
for cls in classes:
    os.makedirs(os.path.join(save_dir, cls), exist_ok=True)

# ================= 2. Standard initial board configuration =================
INITIAL_BOARD = [
    ['10_r_black', '08_n_black', '09_b_black', '11_q_black', '12_k_black', '09_b_black', '08_n_black', '10_r_black'],
    ['07_p_black'] * 8,
    ['00_Empty'] * 8,
    ['00_Empty'] * 8,
    ['00_Empty'] * 8,
    ['00_Empty'] * 8,
    ['01_P'] * 8,
    ['04_R', '02_N', '03_B', '05_Q', '06_K', '03_B', '02_N', '04_R']
]

# ================= 3. Calibration phase =================
clicked_points = []
calibration_frame = None

def mouse_callback(event, x, y, flags, param):
    global clicked_points
    if event == cv2.EVENT_LBUTTONDOWN:
        clicked_points.append([x, y])
        cv2.circle(calibration_frame, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Calibration", calibration_frame)

cap = cv2.VideoCapture('chess_video.mp4')
print("\nPress SPACE to pause/resume. Press 'c' to capture a clear chessboard frame for calibration...")
paused = False
while True:
    if not paused:
        ret, frame = cap.read()
        if not ret: break
    cv2.imshow("Video Player", frame)
    key = cv2.waitKey(30) & 0xFF
    if key == ord(' '): paused = not paused
    elif key == ord('c'):
        calibration_frame = frame.copy()
        break
cv2.destroyWindow("Video Player")

cv2.imshow("Calibration", calibration_frame)
cv2.setMouseCallback("Calibration", mouse_callback)
print("Please click the four actual corners of the chessboard (Order: Top-Left -> Top-Right -> Bottom-Right -> Bottom-Left)")
cv2.waitKey(0)
cv2.destroyAllWindows()

width, height = 800, 800
pts_src = np.array(clicked_points, dtype=np.float32)
pts_dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=np.float32)
matrix = cv2.getPerspectiveTransform(pts_src, pts_dst)

# ================= 4. Automated data collection and cropping =================
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
print("\nReplaying video...")
print("Instruction: Press the 's' key 3 to 5 times between 00:00 and 01:03 when the frame is completely static without hand occlusions.")

frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret: break
        
    curr_warped = cv2.warpPerspective(frame, matrix, (width, height))
    cv2.imshow('Auto Labeler', curr_warped)
    
    key = cv2.waitKey(30) & 0xFF
    if key == 27: break
    elif key == ord('s'):
        frame_count += 1
        sq_size = 100
        saved = 0
        for r in range(8):
            for c in range(8):
                # Extended bounding box logic: expand the upper boundary to preserve the tops of the chess pieces
                y1 = max(0, r * sq_size - 80) 
                y2 = (r + 1) * sq_size
                x1 = max(0, c * sq_size + 5)
                x2 = min(width, (c + 1) * sq_size - 5)
                
                square_img = curr_warped[y1:y2, x1:x2]
                folder_name = INITIAL_BOARD[r][c]
                filename = os.path.join(save_dir, folder_name, f"auto_{frame_count}_r{r}_c{c}.jpg")
                
                cv2.imwrite(filename, square_img)
                saved += 1
        print(f"Extraction {frame_count} completed: 64 squares successfully cropped and classified.")

cap.release()
cv2.destroyAllWindows()

## Execute Module: `02_model_trainer.py`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

# ================= 1. Dataset Preparation =================
data_dir = 'chess_dataset'

if not os.path.exists(data_dir):
    print(f"Error: Directory '{data_dir}' not found. Please verify the dataset path.")
    exit()

# Define image preprocessing pipeline (Resize to 224x224 and convert to tensor)
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    # Apply random color and brightness jitter to enhance model robustness against lighting variations and glare
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load the categorized image directory
dataset = datasets.ImageFolder(data_dir, transform=data_transforms)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)

# Output the inferred class labels based on directory structure
class_names = dataset.classes
print(f"Initializing training for {len(class_names)} classes: \n{class_names}")

# ================= 2. Model Initialization (ResNet-18) =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUtilizing {device} for model training...")

# Load a pre-trained ResNet-18 model to leverage transfer learning
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Modify the final fully connected layer to output the specific number of target classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

# Define the loss function and optimizer (Learning rate: 0.001)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ================= 3. Training Loop =================
num_epochs = 10  # Number of training iterations over the entire dataset

print("\nInitiating training process...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    corrects = 0
    total = 0
    
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad() # Clear gradients from the previous step
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels) # Compute loss
        
        loss.backward() # Backpropagation
        optimizer.step() # Update model weights
        
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        corrects += torch.sum(preds == labels.data)
        total += inputs.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = corrects.double() / total
    
    print(f"Epoch {epoch+1}/{num_epochs} completed -> Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}")

# ================= 4. Model Serialization =================
save_path = 'chess_ai_model.pth'
torch.save(model.state_dict(), save_path)
print(f"\nTraining successfully concluded. Model weights serialized and saved to '{save_path}'")
print("The model is now ready for deployment in the automated extraction pipeline.")

## Execute Module: `03_extract_static_camera.py`

In [ ]:
import cv2
import numpy as np
import chess
import chess.pgn
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os

# ================= Helper Functions =================
def get_square_name(row, col):
    """Convert grid row and column indices to standard algebraic notation (a1-h8)."""
    files = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
    ranks = ['8', '7', '6', '5', '4', '3', '2', '1']
    return files[col] + ranks[row]

# ================= 1. Configuration and Model Initialization =================
VIDEO_PATH = 'chess_video.mp4'
START_TIME_SEC = 54  # Skip the initial video segment without active gameplay
MODEL_PATH = 'chess_ai_model.pth'
CLASSES = ['Empty', 'P', 'N', 'B', 'R', 'Q', 'K', 'p', 'n', 'b', 'r', 'q', 'k']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device).eval()
    print("Success: Model weights loaded successfully.")
else:
    print("Error: Model file not found. Please execute the training script first.")
    exit()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ================= 2. Initial Static Calibration =================
clicked_points = []
def mouse_callback(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        clicked_points.append([x, y])
        cv2.circle(param, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Calibration", param)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
cap.set(cv2.CAP_PROP_POS_FRAMES, START_TIME_SEC * fps)

ret, first_frame = cap.read()
if not ret:
    print("Error: Failed to read the video stream.")
    exit()
    
calib_frame = first_frame.copy()
cv2.imshow("Calibration", calib_frame)
cv2.setMouseCallback("Calibration", mouse_callback, calib_frame)
print("\nPlease select the four physical corners of the chessboard (Order: Top-Left -> Top-Right -> Bottom-Right -> Bottom-Left).")
cv2.waitKey(0)
cv2.destroyWindow("Calibration")

if len(clicked_points) != 4:
    print("Calibration cancelled.")
    exit()

# Compute the static perspective transformation matrix
pts_src = np.array(clicked_points, dtype=np.float32)
pts_dst = np.array([[0, 0], [799, 0], [799, 799], [0, 799]], dtype=np.float32)
M_orig = cv2.getPerspectiveTransform(pts_src, pts_dst)

# ================= 3. Inference Engine Initialization =================
board = chess.Board()
game = chess.pgn.Game()
node = game 

prev_gray = None
stable_frames = 0
is_moving = False

print("\n========================================")
print("Static camera extraction mode initialized.")
print("Interference variables eliminated. Awaiting frame stabilization for inference...")
print("========================================")

# ================= 4. Main Execution Loop =================
while True:
    ret, frame = cap.read()
    if not ret: break
    
    # Apply the static perspective transform to rectify the chessboard image
    curr_warped = cv2.warpPerspective(frame, M_orig, (800, 800))
    curr_gray = cv2.GaussianBlur(cv2.cvtColor(curr_warped, cv2.COLOR_BGR2GRAY), (21, 21), 0)
    
    if prev_gray is None:
        prev_gray = curr_gray
        continue

    # --- Motion Detection (Restricted to the 800x800 rectified board area) ---
    diff = cv2.absdiff(prev_gray, curr_gray)
    thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)[1]
    kernel = np.ones((5, 5), np.uint8)
    dilated = cv2.dilate(cv2.erode(thresh, kernel, iterations=2), kernel, iterations=2)
    
    changed_pixels = cv2.countNonZero(dilated)
    
    # Threshold: Pixel change count > 1500 indicates motion (e.g., hand occlusion)
    if changed_pixels > 1500:
        is_moving = True
        stable_frames = 0
        cv2.putText(curr_warped, "MOTION DETECTED", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
    else:
        stable_frames += 1
        cv2.putText(curr_warped, f"STABLE: {stable_frames}/30", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
        
        # Trigger inference when the frame remains stable for 30 consecutive frames (approx. 1 second) post-motion
        if stable_frames == 30 and is_moving:
            is_moving = False
            print("\nFrame stabilized. Initiating model inference...")
            
            board_probs = {}
            for r in range(8):
                for c in range(8):
                    # Bounding box cropping logic: expand upper boundary to capture the piece tops
                    y1, y2 = max(0, r*100 - 80), (r+1)*100
                    x1, x2 = max(0, c*100 + 5), min(800, (c+1)*100 - 5)
                    sq_img = curr_warped[y1:y2, x1:x2]
                    
                    img_rgb = cv2.cvtColor(sq_img, cv2.COLOR_BGR2RGB)
                    tensor = transform(Image.fromarray(img_rgb)).unsqueeze(0).to(device)
                    with torch.no_grad():
                        out = model(tensor)
                        probs = torch.nn.functional.softmax(out[0], dim=0)
                        board_probs[get_square_name(r, c)] = {CLASSES[i]: probs[i].item() for i in range(len(CLASSES))}
            
            best_move, max_score = None, -1
            for move in board.legal_moves:
                s_sq, e_sq = move.uci()[:2], move.uci()[2:4]
                p_sym = board.piece_at(move.from_square).symbol()
                score = board_probs[s_sq]['Empty'] + board_probs[e_sq][p_sym]
                
                if score > max_score:
                    max_score, best_move = score, move
            
            if max_score > 1.2:
                board.push(best_move)
                node = node.add_variation(best_move)
                print(f"Move inferred: {best_move.uci()} (Confidence score: {max_score:.2f})")

    cv2.imshow('Static Inference View', curr_warped)
    prev_gray = curr_gray
    
    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()

print("\nExtraction complete. Generating PGN file...")
with open("extracted_game_static.pgn", "w", encoding="utf-8") as f:
    exporter = chess.pgn.FileExporter(f)
    game.accept(exporter)
print("Success: Game successfully exported to extracted_game_static.pgn.")